In [1]:
import cv2
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, f1_score
import joblib
import torch
import torch.nn as nn
from torchvision.transforms import Resize, ToTensor, Compose, Normalize, transforms

from tqdm import tqdm
from torchvision import transforms
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from os import listdir
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from keras.layers import Reshape
from keras.layers import Input, Dense, concatenate
from keras.models import Model
from collections import defaultdict
from xgboost import XGBClassifier
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from torchvision.models import mobilenet_v3_large
from sklearn.metrics import precision_score, recall_score
import seaborn as sns
import tensorflow as tf

2026-03-03 10:27:59.488676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772533679.744930      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772533679.819337      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772533680.422728      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772533680.422784      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772533680.422788      17 computation_placer.cc:177] computation placer alr

In [2]:
data = pd.read_csv("/kaggle/input/datasets/sealeopard/brain-tumor-csv/Brain Tumor.csv")
data.head()

,Image,Class,Mean,Variance,Standard Deviation,Entropy,Skewness,Kurtosis,Contrast,Energy,ASM,Homogeneity,Dissimilarity,Correlation,Coarseness
0,Image1,0,6.535339,619.587845,24.891522,0.109059,4.276477,18.900575,98.613971,0.293314,0.086033,0.530941,4.473346,0.981939,7.458341e-155
1,Image2,0,8.749969,805.957634,28.389393,0.266538,3.718116,14.464618,63.858816,0.475051,0.225674,0.651352,3.220072,0.988834,7.458341e-155
2,Image3,1,7.341095,1143.808219,33.820234,0.001467,5.061750,26.479563,81.867206,0.031917,0.001019,0.268275,5.981800,0.978014,7.458341e-155
3,Image4,1,5.958145,959.711985,30.979219,0.001477,5.677977,33.428845,151.229741,0.032024,0.001026,0.243851,7.700919,0.964189,7.458341e-155
4,Image5,0,7.315231,729.540579,27.010009,0.146761,4.283221,19.079108,174.988756,0.343849,0.118232,0.501140,6.834689,0.972789,7.458341e-155


In [3]:
images = data.Image
classes = data.Class
image_label_dict = data.set_index('Image')['Class'].to_dict()

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [5]:
def preprocess_image(image_path):
  image = Image.open(image_path).convert('RGB')
  transform = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      #transforms.Normalize(mean=[0.5], std=[0.5])
  ])
  image = transform(image)
  image = image.unsqueeze(0)
  return image.to(device)
def extract_features(image, model):
  with torch.no_grad():
    feature = model(image).squeeze().cpu().numpy()
  return feature
def extract_features_from_image(image_path, model):
  image = preprocess_image(image_path)
  features = extract_features(image, model)
  return features
def preprocess_image_pca(image_path):
    image = Image.open(image_path).convert('RGB') 
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    image = transform(image)
    image = image.numpy().flatten()
    return image

In [6]:
import torchvision.models as models

mobilenet_v3_large_model = models.mobilenet_v3_large(pretrained=True) 
mobilenet_v3_large_model = nn.Sequential(*list(mobilenet_v3_large_model.children())[:-1])  
mobilenet_v3_large_model = mobilenet_v3_large_model.to(device)
mobilenet_v3_large_model.eval()

dir_list = '/kaggle/input/datasets/sealeopard/reloaded-crop/Crop/'
X_mobilenet_v3_large = []
X_pca_orig = []
Y = []
filename_list = []

for filename in tqdm(listdir(dir_list)):
    if filename.endswith('.jpg'):
        filename_list.append(filename)
        path = dir_list + filename
        label = image_label_dict.get(filename.split('.')[0], None)
        
        if label is not None:
            features = extract_features_from_image(path, mobilenet_v3_large_model)
            features = features.flatten()  
            X_mobilenet_v3_large.append(features)
            feature_pca = preprocess_image_pca(path)
            X_pca_orig.append(feature_pca)
            Y.append(label)

X_resnet = np.array(X_mobilenet_v3_large)
X_pca_orig = np.array(X_pca_orig)
Y = np.array(Y)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 21.1M/21.1M [00:00<00:00, 133MB/s] 
100%|██████████| 3762/3762 [01:52<00:00, 33.30it/s]


In [7]:
print(X_resnet[0].shape)
print(X_pca_orig[0].shape)

(960,)
(150528,)


In [8]:
n_components = 0.95
pca_model = PCA(n_components=n_components, random_state=42)
X_pca = pca_model.fit_transform(X_pca_orig)
print(X_pca[0].shape)
X_pca = np.hstack((X_resnet, X_pca))
print(X_pca[0].shape)

(378,)
(1338,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_pca, Y, test_size=0.5, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
print(X_train.shape)

(1598, 1338)


In [10]:
C_values = [0.1, 0.5, 1, 2, 5, 10, 15, 20, 50]
gamma_values = ['scale', 'auto']

best_f1 = 0
best_params = {}

#scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train)
#X_val = scaler.transform(X_val)
#X_test = scaler.transform(X_test)

for C in C_values:
    for gamma in gamma_values:
        svm_model = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        svm_model.fit(X_train, y_train)
        y_val_pred = svm_model.predict(X_val)
        val_f1 = f1_score(y_val, y_val_pred)  
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_params = {'C': C, 'gamma': gamma}

best_model = SVC(kernel='rbf', C=best_params['C'], gamma=best_params['gamma'], random_state=42)
best_model.fit(X_train, y_train)
y_test_pred = best_model.predict(X_test)

print("Best Hyperparameters:", best_params)
print("Validation F1 Score:", best_f1)
print("Test F1 Score:", f1_score(y_test, y_test_pred))
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("Classification Report:\n", classification_report(y_test, y_test_pred, digits=4))

Best Hyperparameters: {'C': 5, 'gamma': 'scale'}
Validation F1 Score: 0.9578544061302682
Test F1 Score: 0.9448441247002398
Accuracy: 0.951089845826688
Precision: 0.9505428226779252
Recall: 0.9392133492252682
Classification Report:
               precision    recall  f1-score   support

           0     0.9515    0.9607    0.9561      1042
           1     0.9505    0.9392    0.9448       839

    accuracy                         0.9511      1881
   macro avg     0.9510    0.9499    0.9505      1881
weighted avg     0.9511    0.9511    0.9511      1881



In [11]:
import joblib

SAVE_DIR = "/kaggle/working/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(pca_model,  os.path.join(SAVE_DIR, "pca_model.joblib"))
joblib.dump(svm_model,  os.path.join(SAVE_DIR, "svm_model.joblib"))

print("Đã lưu xong PCA, SVM vào:", SAVE_DIR)

metadata = {
    "n_components": n_components,
    "image_size": (224, 224),
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std":  [0.229, 0.224, 0.225],
    "C": best_params['C'], 
    "gamma": best_params['gamma'],
    "classes": sorted(set(data['Class'])),  # hoặc list các class bạn có
    "date_trained": "2025-03"               # thay bằng thời gian thật
}

import json
with open(os.path.join(SAVE_DIR, "metadata_svm.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Đã lưu metadata.json")

Đã lưu xong PCA, SVM vào: /kaggle/working/saved_models
Đã lưu metadata.json
